# Hypothesis Space Interventions

## Group B: what functions the learner can represent

This notebook investigates whether the baseline fails because the model family or input representation makes the needed function hard to express. Data rows, split convention, optimizer, and loss remain fixed.

By the end of this notebook you should be able to:

- distinguish a hypothesis-space intervention from a data or optimizer intervention;
- use train/validation evidence to argue for or against a failure hypothesis;
- choose one model-family change without using the test set as a search tool;
- record a concise experiment result that your group can report back.

**Activity contract**

| Category | Rule |
|---|---|
| Fixed | Data rows, split convention, optimizer, loss, evaluation report |
| Allowed | Width, depth, regularity strength, candidate input set, activation |
| Not allowed | Changing data collection, changing split strategy, changing optimizer, changing loss |
| Selection rule | Choose interventions using validation evidence |
| Test rule | Use test performance only after the choice is fixed |

Keep `show_advanced=False` unless an extension is explicitly enabled.

**Reasoning loop for every subsection**

$$
H_{\mathrm{fail}}
\xrightarrow{E}
I_E
\xrightarrow{\Delta}
H_{\Delta}
\xrightarrow{\mathcal{E}}
R
\xrightarrow{J}
C.
$$

Here $H_{\mathrm{fail}}$ is the failure hypothesis, $E$ is evidence, $I_E$ is the interpretation of that evidence, $\Delta$ is the intervention design, $H_{\Delta}$ is the intervention hypothesis, $\mathcal{E}$ is the experiment, $R$ is the result, $J$ is the evaluation judgement, and $C$ is the conclusion or discussion claim.

Use the result log in each subsection to capture what you tried, what happened, and what you concluded. The goal is not to run a broad hyperparameter search; the goal is to run a controlled test of a modelling explanation.


In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

REPO_URL = "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git"
REPO_BRANCH = "workshop3"
PACKAGE_NAME = "nextgen2026_mlai_workshops"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    missing_packages = [
        package_name
        for package_name, module_name in (("pandas", "pandas"), ("torch", "torch"))
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    repo_dir = None
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / PACKAGE_NAME).exists():
            repo_dir = possible_root
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name == PACKAGE_NAME or module_name.startswith(f"{PACKAGE_NAME}."):
        del sys.modules[module_name]

print(f"Workshop 3 environment ready. Repository: {repo_dir or 'installed environment'}")


Workshop 3 environment ready. Repository: /home/sam/work/nextgen2026-mlai-workshops


In [ ]:
# Load helpers and set the fixed baseline configuration for hypothesis-space comparisons.
from nextgen2026_mlai_workshops import solar_pv as pv
import matplotlib.pyplot as plt
import numpy as np

show_advanced = False
fixed_baseline_config = pv.baseline_config()

def require_filled_options(options, name):
    if any(option is Ellipsis for option in options):
        raise ValueError(
            f"Replace the ... placeholder in {name} with option lines before running this evidence cell."
        )


pv.print_table(
    ["Category", "Setting"],
    [
        ["Fixed rows", "train, validation, and final test rows supplied by the helper"],
        ["Fixed split convention", "train for fitting, validation for choosing, test for final check"],
        ["Fixed optimizer", "plain SGD with the workshop baseline settings"],
        ["Fixed objective", "MSE loss with validation RMSE selection"],
        ["Allowed model-family choices", "width, depth, regularity, input set, activation"],
        ["Advanced diagnostics", show_advanced],
    ],
)


## 1. Capacity and Regularity

### Motivation

The hypothesis space defines which input-output relationships the learner can represent easily. The limiting factor may be that the current model family is either too constrained to capture real structure or too flexible without enough regularity to generalize from finite evidence.

### Failure Hypothesis

The baseline model may have the wrong amount of capacity or regularity. A model that is too small can miss real structure and leave both train and validation errors high; a model that is too flexible can fit training-specific patterns that do not carry over to validation rows.

### Evidence

The next cell keeps the rows, split convention, optimizer, objective, activation, and inputs fixed. It starts from the baseline model and asks your group to add a small set of width/depth/regularity options to compare. Because the data and training recipe stay fixed, the evidence is about the function class and its regularity preference rather than a new data or optimizer choice.


In [ ]:
# Build the capacity scenario and compare model-size options.
print("Regularity options:", ", ".join(pv.REGULARITY_OPTIONS))
print("Width and depth are numeric; keep the comparison small.")

capacity_bundle = pv.make_workshop3_bundle("hypothesis_capacity", seed=7)
capacity_options = (
    {"label": "baseline", "hidden_width": 16, "depth": 2},
    ...  # TODO: Add options to test, e.g. {"label": "tiny", "hidden_width": 4, "depth": 1}
)
require_filled_options(capacity_options, "capacity_options")

capacity_results = pv.run_model_family_options(
    capacity_bundle,
    options=capacity_options,
    seed=7,
    show_advanced=show_advanced,
)

capacity_table_rows = []
for row in capacity_results["rows"]:
    label = row[0]
    report = pv.evaluate_model_report(
        capacity_results["runs"][label],
        capacity_bundle,
        include_test=False,
        show_advanced=show_advanced,
    )
    capacity_table_rows.append([
        *row[:6],
        report["criterion"]["key_range_validation_rmse"],
        row[6],
        report["criterion"]["passes"],
    ])

pv.print_table(
    ["Label", "Width", "Depth", "Regularity", "Train RMSE", "Validation RMSE", "Key range RMSE", "Gap", "Criterion pass"],
    capacity_table_rows,
)

curve_labels = [label for label in ("tiny", "baseline", "wide regularized") if label in capacity_results["runs"]]
if not curve_labels:
    curve_labels = list(capacity_results["runs"])[:3]
for label in curve_labels:
    fig, _ = pv.plot_training_curves(capacity_results["runs"][label])
    fig.suptitle(f"Training curves: {label}", y=1.03)
    plt.show()


### Interpretation and Rationale

Use the table and curves to decide what kind of failure the evidence supports. A capacity explanation is strongest when the train/validation pattern changes in the way the explanation predicts, not merely when one option has the smallest validation number.

High train and validation error suggests the available functions are too constrained or not shaped well for the relationship. Low train error with a growing validation gap suggests the model can fit the training set but lacks the right regularity. Small differences across options suggest capacity may not be the main limiting factor.

Ask:

- Are both train and validation errors high, suggesting a constrained function class?
- Is the train-validation gap large, suggesting a more flexible model is fitting patterns that do not generalize?
- Does regularity improve validation behaviour without changing the data or optimizer?
- Are the capacity differences large enough to support a modelling story, or are they modest compared with seed and slice variation?

### Intervention Hypothesis

State the model-family story before you select an option. For example, your hypothesis might be that a wider model should help because the input-output relationship has localized nonlinear structure, or that regularity should help because a large model is fitting too much training-specific detail.

### Experiment

The starter cell begins with the baseline and a TODO example. Add a small number of options to `capacity_options`, keeping the same fixed rows, optimizer, objective, and evaluation report. Choose from the option labels using validation evidence, then set `selected_capacity_label` in the next cell.


In [ ]:
# Record the capacity option selected from validation evidence.
selected_capacity_label = None

if selected_capacity_label is None:
    print("Set selected_capacity_label to one option after reviewing validation evidence.")
    print("Available options:", ", ".join(capacity_results["runs"].keys()))
elif selected_capacity_label not in capacity_results["runs"]:
    print("Unknown option. Available options:", ", ".join(capacity_results["runs"].keys()))
else:
    selected_capacity_run = capacity_results["runs"][selected_capacity_label]
    print(f"Validation report for selected capacity/regularity option: {selected_capacity_label}")
    pv.print_report(
        pv.evaluate_model_report(
            selected_capacity_run,
            capacity_bundle,
            include_test=False,
            show_advanced=show_advanced,
        )
    )
    print(f"\nFinal check for selected capacity/regularity option: {selected_capacity_label}")
    pv.print_report(pv.final_check(selected_capacity_run, capacity_bundle, show_advanced=show_advanced))


### Evaluation

Use validation RMSE, key-range validation RMSE, train RMSE, and the train-validation gap to justify the chosen option. The final-check cell prints test results only after `selected_capacity_label` is set.

### Conclusion and Discussion

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

Discussion prompt: would simply making the model as large as possible always help? Why or why not?

Drill deeper:

- Did your selected option improve the failure pattern you diagnosed, or just win a metric by a small amount?
- If train and validation both improve, what evidence would still be needed to rule out optimizer effects?
- If regularity helps, what does that say about the kinds of functions the model was preferring before?


## 2. Additional Input Measurements

### Motivation

Sometimes the limiting factor is not model size but missing information in the input representation. If two examples look the same under the original inputs but require different outputs, no amount of width or depth can reliably separate them unless the model receives a variable that distinguishes the cases.

### Failure Hypothesis

The original input set may not allow the model to express behaviour that depends on measured conditions it does not receive. The model may be forced to average over different cloud or panel-temperature states that are collapsed into the same original input coordinates.

This section is the only point in the activity where additional candidate measurements are revealed. The candidate measurements are `cloud_cover` and `panel_temperature`. Candidate sensor C is also available for comparison.

### Evidence

The next cell uses the same rows and split convention. It shows examples where rows are close under the original inputs but have different target values, then summarizes baseline residuals by the newly revealed candidate measurements. The evidence asks whether residual structure aligns with information that was unavailable to the baseline model.


In [ ]:
# Build the candidate-measurement scenario and inspect residual evidence.
INPUT_COLUMNS = ["irradiance", "ambient_temperature", "tilt_angle"]
measurement_bundle = pv.make_workshop3_bundle("hypothesis_new_measurements", seed=7)
measurement_baseline = pv.train_with_config(
    measurement_bundle,
    fixed_baseline_config,
    name="input baseline",
)

validation = measurement_bundle["validation"]
X_input = pv.feature_matrix(validation, INPUT_COLUMNS)
input_scaler = pv.fit_scaler(X_input, "standard")
X_input_z = pv.transform_with_scaler(X_input, input_scaler)
y_validation = pv.target_vector(validation)

pair_candidates = []
for i in range(len(y_validation)):
    distances = np.sqrt(np.sum((X_input_z - X_input_z[i]) ** 2, axis=1))
    distances[i] = np.inf
    j = int(np.argmin(distances))
    pair_candidates.append((abs(float(y_validation[i] - y_validation[j])), float(distances[j]), i, j))

seen = set()
similar_rows = []
for target_gap, input_distance, i, j in sorted(pair_candidates, reverse=True):
    key = tuple(sorted((i, j)))
    if key in seen:
        continue
    seen.add(key)
    similar_rows.append([
        f"{i}/{j}",
        input_distance,
        target_gap,
        float(validation["cloud_cover"][i]),
        float(validation["cloud_cover"][j]),
        float(validation["panel_temperature"][i]),
        float(validation["panel_temperature"][j]),
    ])
    if len(similar_rows) == 6:
        break

print("Similar under original inputs, different in target")
pv.print_table(
    ["Rows", "Input distance", "Target gap", "Cloud A", "Cloud B", "Panel temp A", "Panel temp B"],
    similar_rows,
)

print("\nResidual summaries by newly revealed candidate measurements")
for measurement, rows in pv.residual_by_candidate_measurement(measurement_bundle, measurement_baseline).items():
    print(f"\n{measurement}")
    pv.print_table(["Bin", "n", "Mean residual", "Mean absolute residual"], rows)


Similar under original inputs, different in target
| Rows    | Input distance | Target gap | Cloud A | Cloud B | Panel temp A | Panel temp B |
| :------ | -------------: | ---------: | ------: | ------: | -----------: | -----------: |
| 140/36  |         0.5300 |     0.2979 |  0.0528 |  0.9309 |      61.5137 |      52.9760 |
| 197/55  |         0.5436 |     0.2674 |  0.8887 |  0.0928 |      47.0107 |      60.1040 |
| 22/191  |         0.3048 |     0.2646 |  0.3424 |  0.9413 |      56.4242 |      52.2371 |
| 8/4     |         0.4323 |     0.2638 |  0.7774 |  0.0112 |      40.4557 |      46.0270 |
| 191/164 |         0.2024 |     0.2507 |  0.9413 |  0.3494 |      52.2371 |      53.6413 |
| 199/100 |         0.5789 |     0.2261 |  0.6166 |  0.0354 |      44.2984 |      49.1321 |

Residual summaries by newly revealed candidate measurements

cloud_cover
| Bin       | n   | Mean residual | Mean absolute residual |
| :-------- | ---: | ------------: | ---------------------: |
| 0.00-0.20 |  8

### Interpretation and Rationale

Treat the table as evidence about input information, not as proof of a physical mechanism. If two rows look similar to the original model but differ in target value, a model using only the original inputs may be forced to average over a missing condition.

A useful candidate input should do two things: line up with a residual pattern before the intervention, and improve validation after the input set changes. If a variable looks correlated with residuals but does not improve validation, it may be redundant, noisy, or not usable by this training setup.

Ask:

- Which newly revealed measurement lines up with residual structure?
- Do similar-original-input examples become easier to distinguish once the candidate measurement is included?
- Does the residual pattern suggest the model needs a new input, or could it be explained by capacity alone?
- Does a candidate input improve validation evidence rather than just making the training setup more complex?

### Intervention Hypothesis

Expanding the input dimension with a useful measured variable should reduce structured residuals because the function class can now condition on information that was unavailable to the baseline.

### Experiment

Complete a small input-set comparison below. The rows are unchanged; only the model input columns change. After reviewing validation evidence, set `selected_input_set` in the final-check cell.


In [ ]:
# Compare candidate input sets with rows and training recipe fixed.
print("Candidate extra input columns:", ", ".join([*pv.REVEALED_INPUTS, pv.CANDIDATE_SENSOR_C]))

INPUT_COLUMNS = ["irradiance", "ambient_temperature", "tilt_angle"]
input_sets = (
    ("baseline", INPUT_COLUMNS),
    ...  # TODO: Add input sets, e.g. ("add cloud", [*INPUT_COLUMNS, "cloud_cover"])
)
require_filled_options(input_sets, "input_sets")

input_results = pv.run_input_set_options(
    measurement_bundle,
    input_sets=input_sets,
    seed=7,
    show_advanced=show_advanced,
)

pv.print_table(
    ["Input set", "Input dim", "Validation RMSE", "Key range RMSE", "Criterion pass"],
    input_results["rows"],
)


In [ ]:
# Record the input set selected from validation evidence.
selected_input_set = None

if selected_input_set is None:
    print("Set selected_input_set to one option after reviewing validation evidence.")
    print("Available options:", ", ".join(input_results["runs"].keys()))
elif selected_input_set not in input_results["runs"]:
    print("Unknown option. Available options:", ", ".join(input_results["runs"].keys()))
else:
    selected_input_run = input_results["runs"][selected_input_set]
    print(f"Validation report for selected input set: {selected_input_set}")
    pv.print_report(
        pv.evaluate_model_report(
            selected_input_run,
            measurement_bundle,
            include_test=False,
            show_advanced=show_advanced,
        )
    )
    print(f"\nFinal check for selected input set: {selected_input_set}")
    pv.print_report(pv.final_check(selected_input_run, measurement_bundle, show_advanced=show_advanced))


### Evaluation

Use validation RMSE, key-range validation RMSE, and the residual-by-measurement summaries to justify the chosen input set. The final-check cell prints test results only after `selected_input_set` is set.

### Conclusion and Discussion

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

Discussion prompt: if adding a variable improves validation, does that prove the model has learned the physical mechanism?

Drill deeper:

- Did the useful variable reduce the same residual pattern that motivated adding it?
- Did candidate sensor C help, and what does that imply about "more inputs" as a general rule?
- What evidence would distinguish genuine mechanism learning from a predictive shortcut?


## 3. Activation and Smoothness Bias

### Motivation

Activation functions do more than make the network nonlinear; they make some shapes easier to represent than others. The limiting factor may be shape bias: the baseline ReLU network represents piecewise-linear surfaces, while PV power often changes smoothly over operating conditions and may be awkward to approximate with the fixed width/depth used here.

### Failure Hypothesis

The activation family changes which function shapes are natural for the model to represent. The baseline ReLU MLP may need more pieces to represent a smooth response, while other activations may introduce different smoothness or saturation biases.

### Evidence

The next cell keeps rows, inputs, width, depth, regularity, optimizer, and objective fixed. It varies activation only, prints validation metrics, and plots prediction sweeps through a representative validation point. The evidence asks whether the learned function shape looks plausible and whether validation supports the same story.


In [ ]:
# Build the activation scenario and compare activation choices.
print("Activation options:", ", ".join(pv.ACTIVATION_OPTIONS))

INPUT_COLUMNS = ["irradiance", "ambient_temperature", "tilt_angle"]
activation_bundle = pv.make_workshop3_bundle("hypothesis_activation", seed=7)
activation_options = (
    "relu",
    ...  # TODO: Add activations to test, e.g. "silu" or "tanh"
)
require_filled_options(activation_options, "activation_options")

activation_results = pv.run_activation_options(
    activation_bundle,
    activations=activation_options,
    seed=7,
    show_advanced=show_advanced,
)

pv.print_table(
    ["Activation", "Initialization", "Validation RMSE", "Key range RMSE", "Criterion pass"],
    activation_results["rows"],
)

validation = activation_bundle["validation"]
representative = {column: float(np.median(validation[column])) for column in INPUT_COLUMNS}
sweep_values = np.linspace(0.0, 1000.0, 120)
sweep_data = {
    column: np.full_like(sweep_values, representative[column], dtype=float)
    for column in INPUT_COLUMNS
}
sweep_data["irradiance"] = sweep_values

fig, ax = plt.subplots(figsize=(7, 4))
sanity_rows = []
for activation, run in activation_results["runs"].items():
    pred = pv.predict_run(run, sweep_data)
    ax.plot(sweep_values, pred, label=activation)
    adjacent_change = np.abs(np.diff(pred))
    sanity_rows.append([
        activation,
        float(np.min(pred)),
        float(np.max(pred)),
        float(np.percentile(adjacent_change, 95)),
        float(np.max(adjacent_change)),
    ])

ax.set_xlabel("irradiance sweep with other inputs held at validation medians")
ax.set_ylabel("predicted normalized power")
ax.set_title("Prediction sweep by activation")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

pv.print_table(
    ["Activation", "Min pred", "Max pred", "95th step change", "Max step change"],
    sanity_rows,
)


### Interpretation and Rationale

Prediction sweeps are slices through the function, not complete proof of behaviour everywhere. Use them with the validation table and the simple prediction-range checks.

A smooth activation is not automatically better. It helps only if the limiting factor is the shape bias of the function class under the fixed training setup. A saturating activation can make some regions too flat, while a piecewise-linear activation can look jumpy or require more width to track smooth curvature.

Ask:

- Do sweeps show overly flat regions, abrupt jumps, or implausible prediction ranges?
- Does validation performance support the same story as the visual sweep?
- Are the differences most visible in the operating region that matters for the criterion?
- Is the activation change testing a function-shape hypothesis rather than simply trying another option?

### Intervention Hypothesis

A smoother or better-matched activation can improve the learned function without changing the data, inputs, optimizer, or objective. The rationale is that the same fixed-size network may represent the relevant response with less awkward shape bias.

### Experiment

The starter cell begins with `relu` and leaves the remaining activation choices for your group. Add a small number of options to `activation_options`. Choose from the option labels using validation evidence, then set `selected_activation` in the next cell.


In [ ]:
# Record the activation selected from validation evidence.
selected_activation = None

if selected_activation is None:
    print("Set selected_activation to one option after reviewing validation evidence.")
    print("Available options:", ", ".join(activation_results["runs"].keys()))
elif selected_activation not in activation_results["runs"]:
    print("Unknown option. Available options:", ", ".join(activation_results["runs"].keys()))
else:
    selected_activation_run = activation_results["runs"][selected_activation]
    print(f"Validation report for selected activation: {selected_activation}")
    pv.print_report(
        pv.evaluate_model_report(
            selected_activation_run,
            activation_bundle,
            include_test=False,
            show_advanced=show_advanced,
        )
    )
    print(f"\nFinal check for selected activation: {selected_activation}")
    pv.print_report(pv.final_check(selected_activation_run, activation_bundle, show_advanced=show_advanced))


### Evaluation

Use validation metrics, the sweep plot, and the simple prediction-range checks together. The final-check cell prints test results only after `selected_activation` is set.

### Conclusion and Discussion

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

Discussion prompt: would a sinusoidal activation help just because this is a solar problem? What input would need to be present for periodic structure to make sense?

Drill deeper:

- Did the activation change alter the sweep in a way that matches the failure hypothesis?
- If validation improves but the sweep looks less plausible, which evidence should carry more weight for this task?
- What would you hold fixed in a follow-up so the activation remains the main tested lever?
